# Run a synthetic focus group in Claude Code via the SynthPanel MCP server

**SynthPanel** is an open-source (MIT) research harness that runs synthetic focus groups using AI personas. You define the people you want to hear from and the questions you want to ask, and SynthPanel runs the panel in parallel and synthesizes the findings.

This cookbook shows two ways to use it with Claude:

1. **Inside Claude Code** (or Cursor / Windsurf / Zed / VS Code) via the bundled **MCP server** — Claude can discover the panel tools and drive the whole study for you with a natural-language prompt.
2. **Programmatically** from this notebook using the Python SDK — the same machinery the MCP server uses, exposed as normal Python functions.

### When is this useful?

- **Sanity-check product messaging** before you put it in front of real humans.
- **Explore pricing and positioning** with diverse personas in minutes, not weeks.
- **Generate discussion guides** by running a cheap synthetic pilot first.
- **Red-team product copy** with skeptical personas.

Synthetic panels are not a replacement for real user research. They are a fast, cheap first pass that helps you write better questions and find the right humans to ask.

## Prerequisites

- Python 3.10+
- An Anthropic API key exposed as the `ANTHROPIC_API_KEY` environment variable.

SynthPanel is LLM-agnostic — it also supports OpenAI, Gemini, and xAI — but we'll use Claude throughout this notebook. All the bundled examples below default to `claude-haiku-4-5` to keep this notebook fast and cheap (well under $0.10 end-to-end).

In [ ]:
%pip install --quiet 'synthpanel[mcp]'

In [ ]:
import os

# Load from environment; never hard-code keys in notebooks.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY before running this notebook."

## Part 1 — Drive a focus group from Claude Code via MCP

The fastest way to run a panel is to let Claude do it for you. SynthPanel ships an [MCP](https://modelcontextprotocol.io) server that exposes 12 tools — `run_prompt`, `run_panel`, `run_quick_poll`, `extend_panel`, persona and instrument pack management, and so on.

Once registered, Claude Code (or any MCP-capable client) can call these tools directly when you ask for a focus group in natural language.

### Register the server with Claude Code

Add SynthPanel to your Claude Code MCP config (`~/.claude/mcp_settings.json` or the project-scoped equivalent):

```json
{
  "mcpServers": {
    "synth_panel": {
      "command": "synthpanel",
      "args": ["mcp-serve"],
      "env": {
        "ANTHROPIC_API_KEY": "sk-ant-..."
      }
    }
  }
}
```

Cursor, Windsurf, and Zed use the same stdio transport — drop the same JSON into their MCP configs.

### Ask Claude to run a panel

With the server registered, you can now ask Claude Code things like:

> *"Run a quick poll on three PM personas about whether they'd pay $49/month for an AI-powered retro facilitator. Summarize the themes."*

Claude will call `run_quick_poll` under the hood, parallelize across the personas, and return a synthesized writeup. For structured studies, ask for a **panel run** and Claude will pick an instrument pack (`pricing-discovery`, `feature-prioritization`, `churn-diagnosis`, etc.) or construct one inline.

### The built-in skill

If you install the Claude Code plugin with `/plugin install synthpanel`, you also get a `/focus-group` slash command that wraps the common flow (define personas → pick instrument → run → interpret).

The rest of this notebook uses the **Python SDK** directly so you can see exactly what the MCP tools call under the hood. The two surfaces accept the same personas and instruments, and return the same result shapes.

## Part 2 — Single-prompt sanity check

Before building a full panel, confirm the client is wired up correctly with a one-shot `run_prompt`. This is the cheapest possible round-trip and costs a fraction of a cent.

In [ ]:
from synth_panel import run_prompt

result = run_prompt(
    "In one sentence, define a synthetic focus group.",
    model="claude-haiku-4-5",
)

print(result.response)
print("---")
print(f"model: {result.model}")
print(f"cost:  {result.cost}")

## Part 3 — A quick poll across three personas

`quick_poll` is the workhorse for single-question studies: three to eight personas, one question, parallel execution, and an automatic synthesis pass that extracts themes, agreements, and disagreements.

We'll ask three very different personas — a time-boxed PM, a skeptical SMB owner, and a budget-conscious grad student — the same pricing question and watch the model surface the tension between them.

In [ ]:
from synth_panel import quick_poll

personas = [
    {
        "name": "Sarah Chen",
        "age": 34,
        "occupation": "Product Manager at a mid-size SaaS company",
        "background": (
            "Eight years in tech. Previously a software engineer. Manages a team "
            "of five and is chronically short on time."
        ),
        "personality_traits": ["analytical", "pragmatic", "detail-oriented"],
    },
    {
        "name": "Marcus Johnson",
        "age": 52,
        "occupation": "Owner of a three-location family restaurant chain",
        "background": (
            "Not very tech-savvy but recognizes the need for digital tools. "
            "Values simplicity and reliability over features."
        ),
        "personality_traits": ["practical", "skeptical of technology"],
    },
    {
        "name": "Priya Sharma",
        "age": 28,
        "occupation": "PhD candidate in computational linguistics",
        "background": (
            "Heavy user of developer tools and APIs. Budget-conscious but "
            "willing to pay for tools that save significant time."
        ),
        "personality_traits": ["curious", "technically sophisticated", "cost-conscious"],
    },
]

poll = quick_poll(
    question=(
        "Imagine a tool that runs AI-moderated retrospectives for your team "
        "or project. It's $49/month per seat. Would you pay for it? Why or why not?"
    ),
    personas=personas,
    model="claude-haiku-4-5",
)

print(f"Ran poll across {len(poll.responses)} personas.")
print(f"Total cost: {poll.total_cost}")

### Individual panelist answers

Each persona speaks for themselves — this is the raw material the synthesizer will later aggregate. `poll.responses[i]["responses"]` is a list of question/answer pairs; for a single-question poll there's exactly one entry per panelist.

In [ ]:
for row in poll.responses:
    answer = row["responses"][0]["response"]
    print(f"\n=== {row['persona']} ===")
    print(answer)

### Synthesis — themes, agreements, disagreements

The synthesis pass runs a second Claude call that reads every panelist response and returns a structured writeup. The dict has `summary`, `themes`, `agreements`, `disagreements`, `surprises`, and `recommendation`. This is what you share with the team or paste into a PRD.

In [ ]:
synth = poll.synthesis or {}

print("SUMMARY\n", synth.get("summary", ""))
print("\nTHEMES")
for t in synth.get("themes", []):
    print(f"  - {t}")
print("\nAGREEMENTS")
for a in synth.get("agreements", []):
    print(f"  - {a}")
print("\nDISAGREEMENTS")
for d in synth.get("disagreements", []):
    print(f"  - {d}")
print("\nRECOMMENDATION\n", synth.get("recommendation", ""))

## Part 4 — A branching v3 panel that picks its own probe

Real focus groups rarely follow a fixed script — a skilled moderator listens to the first round of answers and decides whether to dig into pain, price, or alternatives. SynthPanel's **v3 branching instruments** let you encode that moderator logic as a small DAG: every round has optional `route_when` predicates that pick the next round based on the themes the synthesizer extracts.

`pricing-discovery` is one of the instrument packs bundled with the library. It starts with a broad discovery round, then routes into one of three probe rounds based on what surfaced.

In [ ]:
from synth_panel import list_instruments, run_panel

# Inspect the bundled instrument packs.
for pack in list_instruments():
    desc = (pack.get("description") or "").strip().replace("\n", " ")
    print(f"- {pack['name']}: {desc[:90]}")

In [ ]:
panel = run_panel(
    personas=personas,
    instrument_pack="pricing-discovery",
    model="claude-haiku-4-5",
)

print(f"Rounds executed: {len(panel.rounds)}")
print(f"Total cost:      {panel.total_cost}")
print("\nRouting path:")
for step in panel.path:
    print(f"  {step['round']} --[{step['branch']}]--> {step['next']}")

### Inspect what each round asked and how the router decided

`panel.rounds` is an ordered list of the rounds the router actually ran (dead-end rounds are not included). Each round has its name, every panelist's response, and the synthesized themes that drove the next routing decision.

In [ ]:
for round_result in panel.rounds:
    print(f"\n=== Round: {round_result['name']} ===")
    themes = (round_result.get("synthesis") or {}).get("themes", [])
    if themes:
        print("Themes extracted:")
        for t in themes:
            print(f"  - {t}")

### The final cross-round synthesis

After the last round runs, SynthPanel produces a *cross-round* synthesis that reasons about the arc of the conversation — not just the final answers.

In [ ]:
final = panel.synthesis or {}

print("SUMMARY\n", final.get("summary", ""))
print("\nKEY THEMES")
for t in final.get("themes", []):
    print(f"  - {t}")
print("\nRECOMMENDATION\n", final.get("recommendation", ""))

## What you just did

In three SDK calls you:

1. Ran a one-shot sanity prompt.
2. Polled three synthetic personas about a pricing question and got an aggregated writeup.
3. Ran a branching focus group where the moderator logic picked its own probe path based on what surfaced in the first round.

Every function you called here is also available as an MCP tool — meaning Claude Code, Cursor, Windsurf, or any MCP-capable agent can drive the exact same study with a natural-language request once you register the `synth_panel` server.

### Next steps

- **Bring your own personas.** YAML-define them (`personas:` at the top level) and load them with `pack_id="..."` or paste them inline.
- **Author your own v3 instrument.** Three predicate ops (`contains`, `equals`, `matches`) and an `__end__` sentinel are enough to model most moderator flows. See the [instrument authoring docs](https://github.com/DataViking-Tech/SynthPanel/tree/main/docs).
- **Pair with real research.** Use synthetic panels to sharpen your discussion guide, then run it with humans. The synthetic pass is cheap; the human pass is where truth lives.
- **Add persona robustness.** Set `variants=5` on `run_panel` to generate perturbations of each persona and get a robustness score — useful when a stakeholder asks *"what if you had used slightly different personas?"*

### Links

- Repo: <https://github.com/DataViking-Tech/SynthPanel>
- Docs: <https://synthpanel.dev>
- MCP spec: <https://modelcontextprotocol.io>
- License: MIT